In [10]:
import shutil
import time
import urllib.request
from pathlib import Path

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, RandomSampler
from torch import nn
from torch.utils.tensorboard import SummaryWriter

In [11]:
SEED = 1337
torch.manual_seed(SEED)

try:
    ROOT = Path(__file__).resolve().parent
except NameError:  # exécution dans une cellule Jupyter : `__file__` n'existe pas
    ROOT = Path.cwd()

DATA_DIR = ROOT / "data"
RUNS_DIR = ROOT / "runs" / "06_transformer"
CHECKPOINT_DIR = ROOT / "checkpoints"
CORPUS_PATH = DATA_DIR / "input.txt"
SHAKESPEARE_URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"

device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"device : {device}")

device : mps


## 1. Données et lots

Vocabulaire caractère, `encode` / `decode` et découpage 90 / 10 : repris
tels quels du script 01.

In [12]:
if not CORPUS_PATH.exists():
    CORPUS_PATH.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(SHAKESPEARE_URL, CORPUS_PATH)
text = CORPUS_PATH.read_text(encoding="utf-8")

chars = sorted(set(text))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}


def encode(s: str) -> list[int]:
    """Texte → liste d'indices."""
    return [stoi[c] for c in s]


def decode(ids: list[int]) -> str:
    """Liste d'indices → texte."""
    return "".join(itos[i] for i in ids)


data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]

# Hyperparamètres communs à tous les modèles du script.
batch_size = 32
block_size = 64
n_embd = 128
learning_rate = 1e-3
eval_every = 250  # on mesure la validation tous les 250 pas
val_batches = 100  # lots de validation par mesure

print(f"corpus : {len(text):,} caractères   vocab_size : {vocab_size}")

corpus : 1,115,394 caractères   vocab_size : 65


Au script 01, une fonction `get_batch` tirait des positions au hasard et
découpait les lots à la main. On fait ici comme en séance 1 avec MNIST : un
**`Dataset`** décrit **un** exemple, et un **`DataLoader`** s'occupe d'en
faire des lots.

Un exemple, c'est un morceau de `block_size` caractères `x`, et le même
décalé d'un cran `y` (les cibles). Il en commence un à chaque position du
texte : le `Dataset` a donc presque autant d'exemples que de caractères.

In [13]:
class CharDataset(Dataset):
    """Tous les morceaux de `block_size` caractères d'un texte, avec leurs cibles.

    Args:
        data: Le texte encodé, un tenseur d'indices.
        block_size: Longueur des morceaux.
    """

    def __init__(self, data: torch.Tensor, block_size: int) -> None:
        self.data = data
        self.block_size = block_size

    def __len__(self) -> int:
        return len(self.data) - self.block_size

    def __getitem__(self, i: int) -> tuple[torch.Tensor, torch.Tensor]:
        chunk = self.data[i : i + self.block_size + 1]
        return chunk[:-1], chunk[1:]  # x, et y décalé d'un cran


train_dataset = CharDataset(train_data, block_size)
val_dataset = CharDataset(val_data, block_size)
x0, y0 = train_dataset[0]
print(f"{len(train_dataset):,} exemples d'entraînement   {len(val_dataset):,} de validation")
print(f"x : {decode(x0[:20].tolist())!r}…")
print(f"y : {decode(y0[:20].tolist())!r}…")

1,003,790 exemples d'entraînement   111,476 de validation
x : 'First Citizen:\nBefor'…
y : 'irst Citizen:\nBefore'…


## On compte en pas, pas en époques

En séance 1, on entraînait par **époques** : une époque, c'est un passage
sur tout le jeu d'entraînement. Ce n'est pas comme ça qu'on entraîne un
modèle de langue.

- **Le corpus est trop gros pour être parcouru.** GPT-3 a été entraîné pour
  un budget fixé à l'avance, 300 milliards de tokens. Il n'a vu que 44 % de
  Common Crawl, sa plus grosse source : moins d'une époque.
- **On fixe donc un nombre de pas, et on tire les lots au hasard.** C'est
  ce que fait Karpathy dans nanoGPT (`max_iters`) et dans nanochat (un
  nombre de pas calculé à partir d'un budget de tokens), et ce que fait
  Llama 3 (1,2 million de pas). Dans Hugging Face, c'est `max_steps`.
- **On mesure à intervalle régulier** : tous les `eval_every` pas, une loss
  de validation. Pas besoin d'attendre la fin d'un passage complet.

Notre corpus est petit (1 Mo), mais on fait comme les vrais : un nombre de
pas `max_steps`, des morceaux tirés au hasard **avec remise** dans le
`Dataset` (`RandomSampler`), et une mesure tous les `eval_every` pas.

- `make_train_loader(max_steps)` fournit exactement `max_steps` lots : un par
  pas de gradient ;
- `val_loader` fournit `val_batches` lots de validation pour une mesure.
  Chaque fois qu'on le parcourt, il tire de nouveaux morceaux.

In [14]:
def make_train_loader(max_steps: int) -> DataLoader:
    """Un `DataLoader` qui tire `max_steps` lots au hasard dans le train."""
    sampler = RandomSampler(train_dataset, replacement=True, num_samples=max_steps * batch_size)
    return DataLoader(train_dataset, batch_size=batch_size, sampler=sampler)


val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    sampler=RandomSampler(val_dataset, replacement=True, num_samples=val_batches * batch_size),
)
xb, yb = next(iter(make_train_loader(max_steps=1)))
print(f"un lot : x {tuple(xb.shape)}   y {tuple(yb.shape)}")

un lot : x (32, 64)   y (32, 64)


## La boucle d'entraînement, commune à tous les modèles

On va entraîner cinq modèles et comparer leurs courbes : ils partagent tous
la même boucle. Elle repose sur deux fonctions bien séparées :

- `train_step` : **un** pas d'entraînement sur un lot. Forward (qui calcule
  aussi la loss), backward, step. Le modèle est en mode `train` (dropout
  actif).
- `test` : mesure la loss de validation sans rien apprendre. Mode `eval`
  (dropout coupé), pas de gradient, moyenne sur `val_batches` lots.

`run_training` est la boucle elle-même, sur le modèle de nanoGPT : pour
chaque pas, un `train_step` ; tous les `eval_every` pas, un `test`. La loss
de train notée dans TensorBoard est la moyenne des pas écoulés depuis la
dernière mesure.

Convention reprise de la vidéo : chaque modèle renvoie `(logits, loss)`, la
loss se calcule **dans** le modèle.

In [15]:
def train_step(model: nn.Module, optimizer: torch.optim.Optimizer, X: torch.Tensor, Y: torch.Tensor) -> float:
    """Un pas d'entraînement sur un lot.

    Args:
        model: Modèle renvoyant `(logits, loss)`.
        optimizer: Optimiseur des paramètres du modèle.
        X: Entrées, de forme `(batch_size, block_size)`.
        Y: Cibles, de même forme.

    Returns:
        La loss du lot.
    """
    model.train()  # dropout actif
    _, loss = model(X.to(device), Y.to(device))  # 1. forward, loss comprise
    optimizer.zero_grad(set_to_none=True)
    loss.backward()  # 2. backward
    optimizer.step()  # 3. step
    return loss.item()


@torch.no_grad()
def test(model: nn.Module, loader: DataLoader) -> float:
    """Mesure la loss moyenne sur les lots du loader, sans apprendre.

    Args:
        model: Modèle renvoyant `(logits, loss)`.
        loader: Lots d'évaluation.

    Returns:
        La loss moyenne.
    """
    model.eval()  # dropout coupé
    total = 0.0
    for X, Y in loader:
        _, loss = model(X.to(device), Y.to(device))
        total += loss.item()
    return total / len(loader)


def run_training(model: nn.Module, run: str, max_steps: int) -> dict[str, float]:
    """Entraîne pendant `max_steps` pas, mesure tous les `eval_every` pas.

    Args:
        model: Modèle renvoyant `(logits, loss)`.
        run: Nom du run, sous `runs/06_transformer/`.
        max_steps: Nombre total de pas de gradient.

    Returns:
        Les loss train et val de la dernière mesure.
    """
    torch.manual_seed(SEED)
    log_dir = RUNS_DIR / run
    shutil.rmtree(log_dir, ignore_errors=True)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    train_losses = []  # loss des pas depuis la dernière mesure
    start = time.perf_counter()

    with SummaryWriter(log_dir=str(log_dir)) as writer:
        for step, (X, Y) in enumerate(make_train_loader(max_steps), start=1):
            train_losses.append(train_step(model, optimizer, X, Y))

            if step % eval_every == 0:
                train_loss = sum(train_losses) / len(train_losses)
                val_loss = test(model, val_loader)
                writer.add_scalars("loss", {"train": train_loss, "val": val_loss}, step)
                train_losses = []

    n_params = sum(p.numel() for p in model.parameters())
    print(f"[{run}] {n_params:,} paramètres   {max_steps:,} pas   {time.perf_counter() - start:.0f} s   "
          f"train {train_loss:.4f}   val {val_loss:.4f}")
    return {"train": train_loss, "val": val_loss}


def sample(model: nn.Module, n_chars: int = 300) -> str:
    """Génère `n_chars` caractères à partir d'un retour à la ligne."""
    torch.manual_seed(SEED)
    context = torch.zeros((1, 1), dtype=torch.long, device=device)
    return decode(model.generate(context, max_new_tokens=n_chars)[0].tolist())

## 2. Le bigramme

Le modèle du script 02, écrit comme un réseau : une table `(vocab_size,
vocab_size)` dont la ligne `i` contient directement les logits du caractère
suivant. `generate` est la boucle de génération : prédire, tirer, ajouter,
recommencer. Elle ne changera plus.

Loss de départ attendue : `ln(65) ≈ 4.17`.

In [16]:
class BigramLanguageModel(nn.Module):
    """Chaque token lit directement les logits du suivant dans une table."""

    def __init__(self) -> None:
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx: torch.Tensor, targets: torch.Tensor | None = None):
        logits = self.token_embedding_table(idx)  # (B, T, vocab_size)
        if targets is None:
            return logits, None
        B, T, C = logits.shape
        # cross_entropy attend (N, C) : on aplatit batch et temps.
        loss = F.cross_entropy(logits.view(B * T, C), targets.view(B * T))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx: torch.Tensor, max_new_tokens: int) -> torch.Tensor:
        """Prolonge `idx` de `max_new_tokens` tokens tirés un par un."""
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]  # jamais plus que la fenêtre de contexte
            logits, _ = self(idx_cond)
            probs = F.softmax(logits[:, -1, :], dim=-1)  # seulement la dernière position
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx


bigram = BigramLanguageModel().to(device)
print(f"loss avant entraînement : {bigram(xb.to(device), yb.to(device))[1].item():.4f}")
run_training(bigram, "1_bigram", max_steps=3_000)
print(sample(bigram))

loss avant entraînement : 4.6362
[1_bigram] 4,225 paramètres   3,000 pas   4 s   train 2.6943   val 2.6922

Thi$sntheMI toscthithinhin.
ANombedeaing--I RGu wo'Then dur gjue,ONGBPPablord devirenoverdAn isXDjt ajusacascleCHILet hgknouk,O'dr heaghzjMOLWiSherAllerdndomnfavOfe.
TWtlNDWhpuSis yfQnd esenigond atonthyved I:


AnaturvDooromo'd.Lar.
HMI ak
FLEdiv.ed nn c.
Ek zuth gha ch'CbrCOWVKslll y z,
Fr bl tome


La loss de départ est un peu au-dessus de `ln(65)` : l'initialisation
aléatoire ne donne pas une distribution parfaitement uniforme. Après 3 000
pas, le bigramme est vers 2,7 (il plafonnerait vers 2,5 en s'entraînant plus
longtemps). Il produit des syllabes, des retours à la ligne et des
majuscules plausibles, mais aucun mot. C'est la référence.

## 3. Une tête d'attention
 Trois changements :

1. les tokens ont maintenant un **embedding** de dimension `n_embd`, et une
    couche `lm_head` transforme le résultat en logits ;
2. on ajoute l'**embedding de position** (script 05, chapitre 7) ;
3. entre les deux, une **tête de self-attention** : chaque position va
  chercher de l'information dans son passé.

In [17]:
class Head(nn.Module):
    """Une tête de self-attention masquée (script 05), avec dropout sur les poids."""

    def __init__(self, head_size: int, dropout: float = 0.0) -> None:
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        _, T, _ = x.shape
        k, q = self.key(x), self.query(x)
        wei = q @ k.transpose(-2, -1) * k.shape[-1] ** -0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        wei = self.dropout(F.softmax(wei, dim=-1))
        return wei @ self.value(x)


class SingleHeadLM(BigramLanguageModel):
    """Embeddings de token et de position, une tête d'attention, une tête de sortie."""

    def __init__(self) -> None:
        nn.Module.__init__(self)
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.sa_head = Head(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx: torch.Tensor, targets: torch.Tensor | None = None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)  # (B, T, n_embd)
        pos_emb = self.position_embedding_table(torch.arange(T, device=idx.device))  # (T, n_embd)
        x = tok_emb + pos_emb  # broadcasting : la même position ajoutée à chaque séquence
        x = self.sa_head(x)
        logits = self.lm_head(x)
        if targets is None:
            return logits, None
        return logits, F.cross_entropy(logits.view(B * T, -1), targets.view(B * T))


run_training(SingleHeadLM().to(device), "2_single_head", max_steps=3_000)

[2_single_head] 74,049 paramètres   3,000 pas   21 s   train 2.2697   val 2.3212


{'train': 2.269722710609436, 'val': 2.3211771869659423}

La validation passe sous le bigramme : regarder le passé aide. `generate`
est hérité du bigramme ; le découpage `idx[:, -block_size:]` devient
indispensable, puisque l'embedding de position n'existe que pour
`block_size` positions.

## 4. Multi-têtes et feed-forward

**Plusieurs têtes**: ici 4 têtes de 32, suivies
d'une projection linéaire.

**Feed-forward** : après l'attention, chaque position passe dans un petit
MLP, **seule**. L'attention est la phase de
**communication** (les tokens échangent de l'information), le feed-forward
la phase de **calcul** (chaque token réfléchit à ce qu'il a reçu). La couche
cachée est 4 fois plus large que `n_embd`, comme dans le papier.

Un **bloc** = communication puis calcul. On en empile 4.

In [18]:
class MultiHeadAttention(nn.Module):
    """Plusieurs têtes en parallèle, concaténées puis projetées."""

    def __init__(self, n_head: int, dropout: float = 0.0) -> None:
        super().__init__()
        self.heads = nn.ModuleList(Head(n_embd // n_head, dropout) for _ in range(n_head))
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.dropout(self.proj(out))


class FeedForward(nn.Module):
    """MLP appliqué à chaque position indépendamment."""

    def __init__(self, dropout: float = 0.0) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class Block(nn.Module):
    """Bloc Transformer : attention (communication) puis feed-forward (calcul).

    Args:
        n_head: Nombre de têtes d'attention.
        residual: Ajoute les connexions résiduelles (chapitre 6).
        layernorm: Ajoute une LayerNorm avant chaque sous-couche (chapitre 6).
        dropout: Taux de dropout (chapitre 7).
    """

    def __init__(self, n_head: int, residual: bool = True, layernorm: bool = True, dropout: float = 0.0) -> None:
        super().__init__()
        self.sa = MultiHeadAttention(n_head, dropout)
        self.ffwd = FeedForward(dropout)
        self.ln1 = nn.LayerNorm(n_embd) if layernorm else nn.Identity()
        self.ln2 = nn.LayerNorm(n_embd) if layernorm else nn.Identity()
        self.residual = residual

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.residual:
            x = x + self.sa(self.ln1(x))
            x = x + self.ffwd(self.ln2(x))
        else:
            x = self.sa(self.ln1(x))
            x = self.ffwd(self.ln2(x))
        return x


class GPTLanguageModel(BigramLanguageModel):
    """GPT caractère : embeddings, `n_layer` blocs, LayerNorm finale, tête de sortie."""

    def __init__(self, n_head: int = 4, n_layer: int = 4, residual: bool = True,
                 layernorm: bool = True, dropout: float = 0.0) -> None:
        nn.Module.__init__(self)
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_head, residual, layernorm, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd) if layernorm else nn.Identity()
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx: torch.Tensor, targets: torch.Tensor | None = None):
        B, T = idx.shape
        x = self.token_embedding_table(idx) + self.position_embedding_table(torch.arange(T, device=idx.device))
        x = self.blocks(x)
        logits = self.lm_head(self.ln_f(x))
        if targets is None:
            return logits, None
        return logits, F.cross_entropy(logits.view(B * T, -1), targets.view(B * T))

## 5. Empiler des blocs, naïvement

Quatre blocs, sans rien d'autre : le réseau devient **profond** (une
douzaine de couches linéaires en série). On retrouve le problème du script
04 : le gradient doit traverser toutes ces couches pour atteindre les
embeddings, et il arrive mal.

In [21]:
run_training(GPTLanguageModel(residual=False, layernorm=False).to(device), "3_deep_plain", max_steps=3_000)

[3_deep_plain] 814,401 paramètres   3,000 pas   52 s   train 3.2614   val 3.2886


{'train': 3.2613773794174192, 'val': 3.288555142879486}

Onze fois plus de paramètres que la tête unique, et **pire que le
bigramme** : la loss reste bloquée vers 3,31. Ce chiffre n'est pas un
hasard : c'est l'entropie des fréquences de caractères du corpus (calculée à
part, 3,31). Le modèle a seulement appris quelles lettres sont fréquentes,
sans rien tirer du contexte. Le gradient n'arrive plus jusqu'aux couches
basses. Dans TensorBoard, la courbe `3_deep_plain` est plate presque dès le
début.


## 6. Connexions résiduelles et LayerNorm

Deux ajouts qui ne changent presque pas le nombre de paramètres, et tout le
reste :

- **connexion résiduelle** : `x = x + sa(x)` au lieu de `x = sa(x)`. Chaque
   bloc **ajoute** une correction au lieu de tout remplacer. À la
   rétropropagation, l'addition transmet le gradient **tel quel** : il existe
   une autoroute directe de la loss jusqu'aux embeddings. C'est la même idée
   que la cellule mémoire du LSTM (script 04), et que les ResNet en vision ;
- **LayerNorm** : normalise chaque vecteur (moyenne 0, variance 1, sur ses
   `n_embd` composantes) avant chaque sous-couche. C'est la normalisation de la
   séance 1, par token plutôt que par lot. On la place **avant** la
   sous-couche (*pre-norm*), contrairement au papier original : c'est ce que
   font GPT-2 et les modèles récents, parce que ça entraîne mieux.

In [22]:
run_training(GPTLanguageModel(residual=True, layernorm=True).to(device), "4_residual_layernorm", max_steps=3_000)

[4_residual_layernorm] 816,705 paramètres   3,000 pas   61 s   train 1.4400   val 1.6267


{'train': 1.440008153438568, 'val': 1.6267074477672576}

Le même modèle profond, au même nombre de paramètres, qui apprend enfin :
la validation tombe vers 1,65, loin sous tous les modèles précédents. Ces deux ingrédients sont ce qui permet
d'empiler des dizaines, puis des centaines de blocs dans les vrais LLM.

## 7. Le GPT final

Dernier ingrédient : le **dropout**, sur les poids d'attention,
après la projection et après le feed-forward. Un plus gros modèle qui
s'entraîne plus longtemps finit par réciter son corpus ; le dropout retarde
ce moment. Même modèle, entraîné deux fois plus longtemps.

In [23]:
dropout = 0.1
model = GPTLanguageModel(n_head=4, n_layer=4, dropout=dropout).to(device)
final_losses = run_training(model, "5_gpt_final", max_steps=6_000)
print(sample(model, 500))

[5_gpt_final] 816,705 paramètres   6,000 pas   144 s   train 1.4542   val 1.5999

That not Marion, for me, for the shall steal that tide.

LADY ANNE:
But be gone of this, thou drawn the scatches
I am hurden furning to for the heaving that may counsel,
Nothing in fond esceneral a better-heart
Shall be to omlow.

QUEEN MARGELEY:
And now the king thy scharbit should all till be to disple.

First Citizen:
More flatter, we dishonour in blin have been-for'd.

DUKE VINCENTIO:
But you dare be rest, that four heart.

CAMILLO:
Here's prince to pardon and kills the truth. Fie
is my figh


Validation vers 1,60. Des répliques précédées d'un nom de personnage en
majuscules, des vers, des mots presque tous anglais : le modèle a appris la **forme** d'une pièce de
Shakespeare, sans le sens. Un modèle de 800 000 paramètres entraîné en
quelques minutes sur 1 Mo de texte ne peut pas faire mieux.

## 8. Sauvegarder

On garde les poids **et** tout ce qu'il faut pour s'en servir : les
hyperparamètres (pour reconstruire l'architecture) et le vocabulaire (sans
lui, les indices ne veulent rien dire). Le script 07 recharge ce fichier.

In [ ]:
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
checkpoint = {
    "model_state": model.state_dict(),
    "config": {
        "vocab_size": vocab_size,
        "block_size": block_size,
        "n_embd": n_embd,
        "n_head": 4,
        "n_layer": 4,
        "dropout": dropout,
    },
    "chars": chars,
    "val_loss": final_losses["val"],
}
torch.save(checkpoint, CHECKPOINT_DIR / "gpt_char.pt")
print(f"→ {(CHECKPOINT_DIR / 'gpt_char.pt').relative_to(ROOT)}")